In [1]:
import torch

In [2]:
seq_len = 6
batch_size = 2
padded_seq_len = seq_len + 2

In [3]:
test = torch.randn(batch_size, padded_seq_len, padded_seq_len)

In [4]:
vertex_lens = torch.tensor([seq_len, padded_seq_len])

In [5]:
from utils.fix_probs import masking4 as masking3

In [6]:
mask3, r = masking3(batch_size, padded_seq_len, vertex_lens, device='cpu')

In [7]:
mask3.shape

torch.Size([2, 8, 8])

In [8]:
test = test.masked_fill(~mask3, float('-inf'))

In [9]:
test = torch.log_softmax(test, dim=-1)

In [10]:
test[0]

tensor([[   -inf, -2.8508, -1.1651, -2.2773, -1.1305, -1.5853,    -inf,    -inf],
        [   -inf,    -inf, -1.2998, -1.7018, -0.7989, -2.3517,    -inf,    -inf],
        [   -inf,    -inf,    -inf, -1.1878, -1.7221, -0.6608,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf, -0.9622, -0.4814,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,  0.0000,    -inf,    -inf],
        [-2.6718, -3.5387, -2.8564, -2.4909, -3.0122, -1.6503, -0.8892, -2.2135],
        [-2.6388, -2.7243, -2.7720, -2.7632, -0.7253, -2.6416, -2.5749, -2.2467],
        [-1.9633, -1.8915, -2.3001, -4.4985, -1.6428, -2.4910, -1.4351, -2.4887]])

In [11]:
test[1]

tensor([[   -inf, -2.0567, -3.3355, -1.0476, -1.4654, -1.9918, -2.8147, -2.8406],
        [   -inf,    -inf, -1.4166, -2.6043, -1.1703, -1.8087, -2.0784, -2.4740],
        [   -inf,    -inf,    -inf, -2.1612, -1.9941, -1.7712, -1.0121, -1.5367],
        [   -inf,    -inf,    -inf,    -inf, -1.4318, -1.4671, -2.4534, -0.8107],
        [   -inf,    -inf,    -inf,    -inf,    -inf, -0.5645, -1.2167, -2.0011],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -3.5433, -0.0293],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  0.0000],
        [-2.0362, -2.7490, -3.1656, -0.7325, -2.8786, -2.6684, -2.3280, -2.8218]])

In [12]:
test = test.masked_fill(r, float('-inf'))

In [13]:
test[0]

tensor([[   -inf, -2.8508, -1.1651, -2.2773, -1.1305, -1.5853,    -inf,    -inf],
        [   -inf,    -inf, -1.2998, -1.7018, -0.7989, -2.3517,    -inf,    -inf],
        [   -inf,    -inf,    -inf, -1.1878, -1.7221, -0.6608,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf, -0.9622, -0.4814,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,  0.0000,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf]])

In [14]:
test[1]

tensor([[   -inf, -2.0567, -3.3355, -1.0476, -1.4654, -1.9918, -2.8147, -2.8406],
        [   -inf,    -inf, -1.4166, -2.6043, -1.1703, -1.8087, -2.0784, -2.4740],
        [   -inf,    -inf,    -inf, -2.1612, -1.9941, -1.7712, -1.0121, -1.5367],
        [   -inf,    -inf,    -inf,    -inf, -1.4318, -1.4671, -2.4534, -0.8107],
        [   -inf,    -inf,    -inf,    -inf,    -inf, -0.5645, -1.2167, -2.0011],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -3.5433, -0.0293],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  0.0000],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf]])

In [15]:
classifier_heads = 4
emb_dim = 8

In [16]:
test_emb = torch.randn(batch_size, padded_seq_len, emb_dim)

In [17]:
attn = torch.nn.Linear(emb_dim, emb_dim * 2)

In [18]:
gates = torch.nn.Linear(emb_dim, classifier_heads)

In [19]:
g = gates(test_emb)

In [20]:
g = torch.log_softmax(g, dim=-1)

In [21]:
q, k = attn(test_emb).split(emb_dim, dim=-1)

In [22]:
q.shape

torch.Size([2, 8, 8])

In [23]:
k.shape

torch.Size([2, 8, 8])

In [24]:
q = q.reshape(batch_size, -1, classifier_heads, emb_dim // classifier_heads)

In [25]:
k = k.reshape(batch_size, -1, classifier_heads, emb_dim // classifier_heads)

In [26]:
q.shape

torch.Size([2, 8, 4, 2])

In [27]:
attn_scores = torch.einsum("bicf,bjcf->bijc", q, k) / (emb_dim // classifier_heads) ** 0.5

In [28]:
attn_scores.shape

torch.Size([2, 8, 8, 4])

In [29]:
attn_scores[0][:, :, 0]

tensor([[ 0.3837, -0.0789,  0.3193,  0.1902,  0.0092,  0.2513, -0.1695,  0.3137],
        [ 0.3879, -0.0429,  0.2255,  0.1854,  0.0263,  0.2339, -0.2052,  0.3711],
        [-0.2593,  0.1678, -0.5177, -0.1500,  0.0466, -0.2324,  0.0094, -0.0447],
        [-0.1895,  0.0361, -0.1500, -0.0934, -0.0059, -0.1225,  0.0864, -0.1592],
        [-0.0743, -0.0765,  0.1804, -0.0196, -0.0441,  0.0016,  0.1172, -0.1950],
        [-0.7956, -0.2226,  0.3574, -0.3220, -0.1974, -0.3097,  0.7063, -1.2153],
        [ 0.2691,  0.0441, -0.0385,  0.1148,  0.0524,  0.1219, -0.2103,  0.3655],
        [-0.8209, -0.1803,  0.2385, -0.3415, -0.1809, -0.3466,  0.6835, -1.1818]],
       grad_fn=<SelectBackward0>)

In [30]:
mask3

tensor([[[False,  True,  True,  True,  True,  True, False, False],
         [False, False,  True,  True,  True,  True, False, False],
         [False, False, False,  True,  True,  True, False, False],
         [False, False, False, False,  True,  True, False, False],
         [False, False, False, False, False,  True, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True,  True,  True]],

        [[False,  True,  True,  True,  True,  True,  True,  True],
         [False, False,  True,  True,  True,  True,  True,  True],
         [False, False, False,  True,  True,  True,  True,  True],
         [False, False, False, False,  True,  True,  True,  True],
         [False, False, False, False, False,  True,  True,  True],
         [False, False, False, False, False, False,  True,  True],
         [False, False, False, False, False, False, False,  

In [31]:
attn_scores = attn_scores.masked_fill(~mask3.unsqueeze(-1), float('-inf'))

In [32]:
attn_scores[0][:, :, 1]

tensor([[   -inf, -0.1036,  0.7175,  0.6761,  0.3947, -0.3502,    -inf,    -inf],
        [   -inf,    -inf,  0.5565,  0.4939,  0.2947, -0.2412,    -inf,    -inf],
        [   -inf,    -inf,    -inf, -0.3474, -0.2931, -0.0269,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf, -0.2090,  0.1665,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,  0.0037,    -inf,    -inf],
        [-0.0091,  0.0503, -0.4112, -0.5514, -0.2877,  0.3642,  0.2144,  0.3785],
        [-0.2679, -0.1352,  0.8106,  0.4355,  0.3229, -0.0683,  0.1002, -0.6208],
        [ 0.0693,  0.1082, -0.7950, -0.8682, -0.4820,  0.5068,  0.2557,  0.6938]],
       grad_fn=<SelectBackward0>)

In [33]:
attn_scores[1][:, :, 1]

tensor([[   -inf,  0.0203,  0.2127,  0.7656, -0.1716, -0.1189, -0.1755,  0.6899],
        [   -inf,    -inf, -0.3447,  0.4277, -0.4724, -0.6331,  0.2566,  0.0905],
        [   -inf,    -inf,    -inf, -0.7103,  0.0261, -0.0899,  0.2881, -0.7442],
        [   -inf,    -inf,    -inf,    -inf, -0.4487, -0.4320, -0.1661,  1.0879],
        [   -inf,    -inf,    -inf,    -inf,    -inf, -0.1624,  0.1064, -0.0989],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  0.0882,  0.5656],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  0.8350],
        [ 0.9460, -0.1503,  0.0840,  1.1096, -0.4308, -0.4465, -0.0828,  0.8572]],
       grad_fn=<SelectBackward0>)

In [34]:
attn_scores = torch.log_softmax(attn_scores, dim=2)

In [35]:
attn_scores = attn_scores.masked_fill(r.unsqueeze(-1), float('-inf'))

In [36]:
attn_scores[0][:, :, 0]

tensor([[   -inf, -1.8376, -1.4394, -1.5685, -1.7494, -1.5073,    -inf,    -inf],
        [   -inf,    -inf, -1.3320, -1.3721, -1.5311, -1.3236,    -inf,    -inf],
        [   -inf,    -inf,    -inf, -1.1436, -0.9471, -1.2261,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf, -0.6365, -0.7532,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,  0.0000,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf]],
       grad_fn=<SelectBackward0>)

In [37]:
attn_scores[1][:, :, 1]

tensor([[   -inf, -2.1736, -1.9813, -1.4284, -2.3655, -2.3129, -2.3694, -1.5041],
        [   -inf,    -inf, -2.0998, -1.3273, -2.2275, -2.3882, -1.4985, -1.6645],
        [   -inf,    -inf,    -inf, -2.1550, -1.4187, -1.5347, -1.1567, -2.1890],
        [   -inf,    -inf,    -inf,    -inf, -2.0785, -2.0618, -1.7959, -0.5419],
        [   -inf,    -inf,    -inf,    -inf,    -inf, -1.2161, -0.9473, -1.1526],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -0.9601, -0.4827],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  0.0000],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf]],
       grad_fn=<SelectBackward0>)

In [38]:
attn_scores.shape, g.shape

(torch.Size([2, 8, 8, 4]), torch.Size([2, 8, 4]))

In [39]:
attn_scores = attn_scores + g.unsqueeze(2)

In [40]:
attn_scores.shape

torch.Size([2, 8, 8, 4])

In [49]:
attn_scores[0][:, :, 0]

tensor([[   -inf, -2.4264, -2.0282, -2.1572, -2.3382, -2.0961,    -inf,    -inf],
        [   -inf,    -inf, -2.4734, -2.5136, -2.6726, -2.4650,    -inf,    -inf],
        [   -inf,    -inf,    -inf, -3.2416, -3.0451, -3.3241,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf, -2.4460, -2.5626,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf, -1.8500,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf]],
       grad_fn=<SelectBackward0>)

In [42]:
from utils.logsumexp import logsumexp_infsafe as logsumexp

In [43]:
attn_scores.shape

torch.Size([2, 8, 8, 4])

In [44]:
logsumexp(attn_scores, dim=-1).shape

torch.Size([2, 8, 8, 1])

In [45]:
final = logsumexp(attn_scores, dim=-1)

In [46]:
final = final.squeeze(-1)

In [47]:
final[0]

tensor([[   -inf, -1.9232, -1.4066, -1.4933, -1.7055, -1.5966,    -inf,    -inf],
        [   -inf,    -inf, -1.2631, -1.3267, -1.4802, -1.4950,    -inf,    -inf],
        [   -inf,    -inf,    -inf, -1.1602, -1.1677, -0.9795,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf, -0.8286, -0.5739,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,  0.0000,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf]],
       grad_fn=<SelectBackward0>)

In [48]:
final[1]

tensor([[   -inf, -1.8851, -1.9758, -1.8200, -2.2022, -2.1906, -1.7872, -1.8473],
        [   -inf,    -inf, -1.8038, -1.6687, -1.9566, -1.9216, -1.5820, -1.8736],
        [   -inf,    -inf,    -inf, -1.8351, -1.4689, -1.5036, -1.4050, -1.9483],
        [   -inf,    -inf,    -inf,    -inf, -1.6664, -1.5918, -1.6207, -0.8922],
        [   -inf,    -inf,    -inf,    -inf,    -inf, -1.0823, -1.1978, -1.0236],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -1.0685, -0.4209],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  0.0000],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf]],
       grad_fn=<SelectBackward0>)